# OTPW 3-Month EDA — W261 Final Project Phase 1
**Purpose:** Exploratory Data Analysis on the OTPW 3-month dataset.  
**Team:** Section 3, Group 2

## Step 0: Setup

In [0]:
from pyspark.sql.functions import col, count, when, isnan, avg, sum as spark_sum, lit, desc
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import pandas as pd

# Adding to avoid casting errors when converting weather string columns to numeric
spark.conf.set("spark.sql.ansi.enabled", "false")

## Step 1: Load OTPW and Save to Parquet

In [0]:
# df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")

# # Checkpoint as parquet (run once, then comment out)
# TEAM_DIR = "dbfs:/student-groups/Group_3_2"
# dbutils.fs.mkdirs(TEAM_DIR)
# df_otpw.write.mode("overwrite").parquet(f"{TEAM_DIR}/otpw_3m.parquet")
# print("Parquet saved.")

In [0]:
# Always read from parquet
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
df = spark.read.parquet(f"{TEAM_DIR}/otpw_3m.parquet")

## Step 2: Schema & Dataset Size

In [0]:
# Dataset dimensions
num_rows = df.count()
num_cols = len(df.columns)
print(f"Rows: {num_rows:,}")
print(f"Columns: {num_cols}")

In [0]:
# Full schema
df.printSchema()

## Step 3: Missing Value Analysis

Check the column with many data missing.

In [0]:
null_exprs = [
    count(
        when(col(c).isNull() | (col(c).cast("string") == ""), c)
    ).alias(c)
    for c in df.columns
]
null_counts = df.select(null_exprs).toPandas().T
null_counts.columns = ["null_count"]
null_counts["null_percentage"] = (null_counts["null_count"] / num_rows * 100).round(2)

null_counts = (
    null_counts
    .reset_index()
    .rename(columns={"index": "column_name"})
    .sort_values("null_percentage", ascending=False)
)

high_missing_cols = null_counts[null_counts["null_percentage"] > 50]
print(f"Columns with >50% missing: {len(high_missing_cols)}")
display(high_missing_cols)

In [0]:
from pyspark.sql.functions import rand

high_missing_col_names = high_missing_cols["column_name"].tolist()
sample_100 = df.select(*high_missing_col_names).orderBy(rand()).limit(100)
display(sample_100)

## Step 4: Duplicate Check

In [0]:
total = df.count()
distinct = df.dropDuplicates().count()
print(f"Total rows: {total}")
print(f"Distinct rows: {distinct}")
print(f"Duplicate rows: {total - distinct}")

## Step 5: Target Variable Distribution

Explore both potential research topics:  
- **DEP_DELAY** (continuous, minutes) — for regression  
- **DEP_DEL15** (binary, 1 = delay ≥ 15 min) — for classification

We first want to check if there is any possibility to clean up the data.

A cancelled flight didn't depart so we could not predict the delay of something that never happened.

In [0]:
# Check for cancelled flights to see if we could make data cleaner.
df.groupBy("CANCELLED").count().show()

In [0]:
# Is it true that all cancelled flights will not have DEP_DELAY and DEP_DEL15?
cancelled = df.filter(col("CANCELLED") == 1)
total_cancelled = cancelled.count()
cancelled_with_delay = cancelled.filter(col('DEP_DELAY').isNotNull()).count()
cancelled_with_del15 = cancelled.filter(col('DEP_DEL15').isNotNull()).count()

print(f"Total cancelled flights: {total_cancelled} ({total_cancelled / num_rows * 100:.2f}% of dataset)")
print(f"Cancelled with non-null DEP_DELAY: {cancelled_with_delay} ({cancelled_with_delay / num_rows * 100:.2f}% of dataset)")
print(f"Cancelled with non-null DEP_DEL15: {cancelled_with_del15} ({cancelled_with_del15 / num_rows * 100:.2f}% of dataset)")


In result, only 0.08% of the dataset will have delay data, so we should be safe to filter out the cancelled data.

In [0]:
# Filter out cancelled flights
df = df.filter(col("CANCELLED") == 0)
num_rows = df.count()
print(f"After removing cancelled flights: {num_rows:,} rows remaining")

In [0]:
# Class balance of DEP_DEL15
target_dist = df.groupBy("DEP_DEL15").count().toPandas()
target_dist["percentage"] = (target_dist["count"] / target_dist["count"].sum() * 100).round(2)
print(target_dist)

plt.figure(figsize=(6, 4))
plt.bar(target_dist["DEP_DEL15"].astype(str), target_dist["count"])
plt.title("DEP_DEL15 Class Distribution (3-Month OTPW)")
plt.xlabel("DEP_DEL15 (0=On-time, 1=Delayed ≥15min)")
plt.ylabel("Count")
for i, row in target_dist.iterrows():
    plt.text(i, row["count"], f"{row['percentage']}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()

In [0]:
# Distribution of DEP_DELAY
dep_delay_pd = df.select(col("DEP_DELAY").cast("double").alias("DEP_DELAY")) \
    .filter(col("DEP_DELAY").isNotNull()) \
    .toPandas()

plt.figure(figsize=(10, 4))
plt.hist(dep_delay_pd["DEP_DELAY"], bins=100, edgecolor="black")
plt.title("DEP_DELAY Distribution")
plt.xlabel("Departure Delay (minutes)")
plt.ylabel("Frequency")
plt.axvline(x=15, color="red", linestyle="--", label="15-min threshold")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.hist(dep_delay_pd["DEP_DELAY"].clip(-60, 300), bins=100, edgecolor="black")
plt.title("DEP_DELAY Distribution (Scoped to -60 to 300)")
plt.xlabel("Departure Delay (minutes)")
plt.ylabel("Frequency")
plt.axvline(x=15, color="red", linestyle="--", label="15-min threshold")
plt.legend()
plt.tight_layout()
plt.show()


### DEP_DEL15 Observations

- The dataset shows a **class imbalance**: approximately 80% of flights are on-time (DEP_DEL15 = 0) vs ~20% delayed (DEP_DEL15 = 1).
- For modeling, we should consider strategies to handle imbalance.


### DEP_DELAY Observations

- The distribution is **heavily right-skewed**: the majority of flights depart on time or with minor delays, but a long tail extends to extreme values.
- Most flights cluster between **-15 and +30 minutes**, indicating that early departures and short delays are the norm.
- The 15-minute threshold (red line) sits at the boundary where the distribution begins to significantly drop. Most of the flights are on the left hand side of this line.
- **Extreme outliers** (delays > 300 min) exist but are rare. These are real events and will not be removed for now, for classification (DEP_DEL15), they are anyways labeled as delayed.

## Step 6: Important Numeric Columns Statistics Summary

In [0]:
castable_cols = []
for c in df.columns:
    non_null = df.select(col(c).cast("double").alias(c)).filter(col(c).isNotNull()).count()
    if non_null > num_rows * 0.5:  # at least 50% non-null after casting
        castable_cols.append(c)

print(f"Columns that cast to double with >50% valid: {len(castable_cols)}")
print(castable_cols)


From the above list, we found following useful features:

- Target variables: DEP_DELAY, DEP_DEL15
- Flight features: CRS_DEP_TIME, CRS_ELAPSED_TIME, DISTANCE
- Hourly weather: 
HourlyDryBulbTemperature, HourlyDewPointTemperature,
HourlyWindSpeed, HourlyWindDirection,
HourlyVisibility, HourlyPrecipitation,
HourlyRelativeHumidity,
HourlyStationPressure,

In [0]:
numeric_cols = [
    "DEP_DELAY", "ARR_DELAY", "DISTANCE", "CRS_ELAPSED_TIME", "CRS_DEP_TIME",
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature", "HourlyWindSpeed", "HourlyWindDirection", "HourlyVisibility", "HourlyPrecipitation", "HourlyRelativeHumidity", "HourlyStationPressure"
]

df_num = df.select([col(c).cast("double").alias(c) for c in numeric_cols if c in df.columns])
display(df_num.describe())


From the count, we could see main features have majority of the data available.
From the min and max data, we could confirm the data is not broken.

## Step 7: Important Categorical Features

In [0]:
print("Unique carriers:", df.select("OP_UNIQUE_CARRIER").distinct().count())
df.groupBy("OP_UNIQUE_CARRIER").count().orderBy(desc("count")).show(20)

In [0]:
print("Unique origin airports:", df.select("ORIGIN").distinct().count())
df.groupBy("ORIGIN").count().orderBy(desc("count")).show(20)

## Step 8: Possible Delay Patterns

In [0]:
dow = df.groupBy("DAY_OF_WEEK").agg(
    count("*").alias("total_flights"),
    spark_sum(col("DEP_DEL15").cast("int")).alias("delayed")
).withColumn("delay_rate", (col("delayed") / col("total_flights") * 100).cast("double")) \
 .orderBy("DAY_OF_WEEK").toPandas()

plt.figure(figsize=(8, 4))
plt.bar(dow["DAY_OF_WEEK"].astype(str), dow["delay_rate"])
plt.title("Delay Rate by Day of Week")
plt.xlabel("Day of Week (1=Mon, 7=Sun)")
plt.ylabel("Delay Rate (%)")
plt.tight_layout()
plt.show()

In [0]:
df_hour = df.withColumn("DEP_HOUR", (col("CRS_DEP_TIME").cast("int") / 100).cast("int"))

hourly = df_hour.groupBy("DEP_HOUR").agg(
    count("*").alias("total_flights"),
    spark_sum(col("DEP_DEL15").cast("int")).alias("delayed")
).withColumn("delay_rate", (col("delayed") / col("total_flights") * 100).cast("double")) \
 .orderBy("DEP_HOUR").toPandas()

plt.figure(figsize=(10, 4))
plt.bar(hourly["DEP_HOUR"], hourly["delay_rate"])
plt.title("Delay Rate by Hour of Day")
plt.xlabel("Scheduled Departure Hour")
plt.ylabel("Delay Rate (%)")
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

In [0]:
monthly = df.groupBy("MONTH").agg(
    count("*").alias("total_flights"),
    spark_sum(col("DEP_DEL15").cast("int")).alias("delayed")
).withColumn("delay_rate", (col("delayed") / col("total_flights") * 100).cast("double")) \
 .orderBy("MONTH").toPandas()

plt.figure(figsize=(6, 4))
plt.bar(monthly["MONTH"].astype(str), monthly["delay_rate"])
plt.title("Delay Rate by Month")
plt.xlabel("Month")
plt.ylabel("Delay Rate (%)")
plt.tight_layout()
plt.show()

In [0]:
carrier = df.groupBy("OP_UNIQUE_CARRIER").agg(
    count("*").alias("total_flights"),
    spark_sum(col("DEP_DEL15").cast("int")).alias("delayed")
).withColumn("delay_rate", (col("delayed") / col("total_flights") * 100).cast("double")) \
 .orderBy(desc("delay_rate")).toPandas()

plt.figure(figsize=(12, 4))
plt.barh(carrier["OP_UNIQUE_CARRIER"], carrier["delay_rate"])
plt.title("Delay Rate by Carrier")
plt.xlabel("Delay Rate (%)")
plt.ylabel("Carrier")
plt.tight_layout()
plt.show()

In [0]:
airport = df.groupBy("ORIGIN").agg(
    count("*").alias("total_flights"),
    spark_sum(col("DEP_DEL15").cast("int")).alias("delayed")
).withColumn("delay_rate", (col("delayed") / col("total_flights") * 100).cast("double")) \
 .filter(col("total_flights") >= 1000) \
 .orderBy(desc("delay_rate")).limit(15).toPandas()

plt.figure(figsize=(10, 5))
plt.barh(airport["ORIGIN"], airport["delay_rate"])
plt.title("Top 15 Airports by Delay Rate")
plt.xlabel("Delay Rate (%)")
plt.ylabel("Origin Airport")
plt.tight_layout()
plt.show()

## Step 9: Weather Feature Distributions

In [0]:
weather_cols = ["HourlyVisibility", "HourlyWindSpeed", "HourlyPrecipitation", "HourlyDryBulbTemperature"]
existing_weather = [c for c in weather_cols if c in df.columns]

df_weather_eda = df.select(
    [col(c).cast("double").alias(c) for c in existing_weather]
)

# Filter extreme outliers for wind speed
if "HourlyWindSpeed" in existing_weather:
    df_weather_eda = df_weather_eda.filter(
        (col("HourlyWindSpeed").isNull()) | (col("HourlyWindSpeed") < 200)
    )

weather_pd = df_weather_eda.toPandas()

fig, axes = plt.subplots(1, len(existing_weather), figsize=(4 * len(existing_weather), 5))
if len(existing_weather) == 1:
    axes = [axes]
for ax, c in zip(axes, existing_weather):
    weather_pd[c].dropna().plot.box(ax=ax)
    ax.set_title(c)
plt.suptitle("Weather Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

## Step 10: Data Leakage Check
Identify columns that would NOT be available 2 hours before departure. These must be excluded from features.

In [0]:
leakage_candidates = [
    "DEP_TIME", "DEP_DELAY", "DEP_DEL15", "DEP_DELAY_NEW", "DEP_DELAY_GROUP",
    "TAXI_OUT", "WHEELS_OFF", "WHEELS_ON", "TAXI_IN",
    "ARR_TIME", "ARR_DELAY", "ARR_DEL15", "ARR_DELAY_NEW", "ARR_DELAY_GROUP",
    "ACTUAL_ELAPSED_TIME", "AIR_TIME",
    "CANCELLED", "CANCELLATION_CODE", "DIVERTED",
    "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"
]
found_leakage = [c for c in leakage_candidates if c in df.columns]
print(f"Potential leakage columns found ({len(found_leakage)}):")
for c in found_leakage:
    print(f"  - {c}")


## Step 11: Observations & Notes

- **Dataset size:** 1,401,363 rows x 216 columns total; 1,357,914 rows after removing cancelled flights
- **Class balance:** ~80% on-time (DEP_DEL15 = 0) vs ~20% delayed (DEP_DEL15 = 1), imbalanced, need class weights or F-beta metric
- **High data miss columns (>50%):** ~105 columns, mostly Daily/Monthly weather aggregates and ShortDuration fields, consider to drop those columns
- **Duplicate rows:** 0
- **Cancelled flights:** 43,449 (3.10% of dataset), of which only 1,143 had delay data (0.08%). Decision: **drop all cancelled flights**.
- **Leakage columns to exclude:** 22 columns identified (DEP_TIME, DEP_DELAY, DEP_DEL15, ARR_*, TAXI_*, WHEELS_*, CANCELLED, delay reason codes, etc.), none of these are known 2 hours before departure
- **Usable numeric features:** DISTANCE, CRS_DEP_TIME, CRS_ELAPSED_TIME, and 8 hourly weather columns (temperature, wind, visibility, precipitation, humidity, pressure)
- **Usable categorical features:** OP_UNIQUE_CARRIER (about 14 carriers), ORIGIN (about 300 airports), DEST, DAY_OF_WEEK, MONTH
- **Key patterns:**
  - Delay rates increase throughout the day, peaking around 8PM.
  - Carrier **F9** has the highest delay rate
  - February appears to have the highest delay rate among Q1 months — likely due to winter weather
  - Weather: most conditions are okay (clear skies, light winds, no precipitation), but outlier events (low visibility, high wind, precipitation) are present and likely correlate with delays